# 00 -- Vertical Slice: End-to-End Hello World

**Objective:** Validate that all system components connect correctly before optimising any individual part.

This notebook walks through a minimal end-to-end pipeline:
1. Download a small set of PubMed abstracts
2. Generate embeddings with a biomedical embedding model
3. Index them in Qdrant
4. Submit a clinical query, retrieve relevant documents, and generate a response with an LLM

**Correctness matters more than quality at this stage.** The goal is a working pipeline from end to end.

---

### Prerequisites
- `docker compose up -d` (Qdrant running at localhost:6333)
- `uv sync --extra dev` (dependencies installed)
- `.env` configured (at minimum, NCBI_EMAIL)

In [ ]:
# Quick check that services are running
import os
from dotenv import load_dotenv

load_dotenv()

# Check Qdrant
from qdrant_client import QdrantClient
qdrant = QdrantClient(host="localhost", port=6333)
print(f"Qdrant connected: {qdrant.get_collections()}")

# Check PubMed email
assert os.getenv("NCBI_EMAIL"), "NCBI_EMAIL is not set in .env"
print("NCBI_EMAIL configured")

## Step 1: Download PubMed abstracts

We use BioPython (Entrez) to search and download a small set of abstracts.
In production this will be a full ingestion pipeline, but here we only need 10-20 papers.

In [ ]:
from Bio import Entrez, Medline
from typing import List, Dict

Entrez.email = os.getenv("NCBI_EMAIL")
api_key = os.getenv("NCBI_API_KEY")
if api_key:
    Entrez.api_key = api_key


def fetch_pubmed_abstracts(
    query: str, max_results: int = 15
) -> List[Dict[str, str]]:
    """Search PubMed and return abstracts with metadata."""
    # Fetch IDs
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
    record = Entrez.read(handle)
    handle.close()
    ids = record["IdList"]
    print(f"Found {len(ids)} papers for query: '{query}'")

    if not ids:
        return []

    # Download details
    handle = Entrez.efetch(db="pubmed", id=ids, rettype="medline", retmode="text")
    records = list(Medline.parse(handle))
    handle.close()

    papers = []
    for r in records:
        abstract = r.get("AB", "")
        if abstract:  # Only include papers with an abstract
            papers.append({
                "pmid": r.get("PMID", ""),
                "title": r.get("TI", ""),
                "abstract": abstract,
                "authors": "; ".join(r.get("AU", [])),
                "journal": r.get("JT", ""),
                "year": r.get("DP", "")[:4],
            })

    print(f"Downloaded {len(papers)} papers with abstracts")
    return papers


# Download a sample set of papers
papers = fetch_pubmed_abstracts("type 2 diabetes treatment guidelines 2024", max_results=15)

# Quick preview
for p in papers[:3]:
    print(f"\n[{p['pmid']}] {p['title'][:80]}...")
    print(f"   {p['abstract'][:150]}...")

## Step 2: Generate embeddings

We use a biomedical embedding model to convert abstracts into dense vectors.
We start with `PubMedBERT`; a comparative benchmark across alternatives will be run in Phase 1.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model
# Alternatives to evaluate in Phase 1: "dmis-lab/biobert-base-cased-v1.2", "BAAI/bge-base-en-v1.5"
EMBEDDING_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
print(f"Loading embedding model: {EMBEDDING_MODEL}")

embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Model loaded | Dimension: {embed_model.get_sentence_embedding_dimension()}")

# Generate embeddings for the abstracts
texts = [p["abstract"] for p in papers]
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=8)

print(f"\nGenerated {len(embeddings)} embeddings of dimension {embeddings.shape[1]}")

## Step 3: Index in Qdrant

We create a collection in Qdrant and upload the embeddings together with their metadata.
This is the RAG knowledge base.

In [ ]:
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION = "pubmed_vertical_slice"  # Temporary name for this test run
VECTOR_DIM = embeddings.shape[1]

# Recreate the collection (clears it if it already exists)
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)
print(f"Collection '{COLLECTION}' created (dim={VECTOR_DIM})")

# Upload points with metadata
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "pmid": papers[i]["pmid"],
            "title": papers[i]["title"],
            "abstract": papers[i]["abstract"],
            "authors": papers[i]["authors"],
            "journal": papers[i]["journal"],
            "year": papers[i]["year"],
        },
    )
    for i in range(len(papers))
]

qdrant.upsert(collection_name=COLLECTION, points=points)

info = qdrant.get_collection(COLLECTION)
print(f"{info.points_count} documents indexed in Qdrant")

## Step 4: Semantic search (Retrieval)

We submit a clinical query, convert it to an embedding, and retrieve the most relevant documents.

In [ ]:
# Test query
query = "What are the first-line treatments for type 2 diabetes in patients with cardiovascular risk?"

# Embed the query
query_embedding = embed_model.encode(query).tolist()

# Search Qdrant
TOP_K = 3
results = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_embedding,
    limit=TOP_K,
)

print(f"Query: {query}\n")
print(f"Top {TOP_K} retrieved documents:\n")
for i, hit in enumerate(results.points):
    print(f"  [{i+1}] Score: {hit.score:.4f} | PMID: {hit.payload['pmid']}")
    print(f"      {hit.payload['title'][:100]}")
    print()

## Step 5: Augmented generation (RAG)

We construct a prompt that includes the retrieved documents as context
and ask the LLM to generate a grounded response.

**For this hello world we use a small model** (or a free API).
Phases 2-3 will use the fine-tuned model.

> Note: If you do not have a GPU, you can use a quantised model (GGUF) via llama-cpp-python,
> or the free Hugging Face Inference API.

In [ ]:
def build_augmented_prompt(query: str, retrieved_docs: list) -> str:
    """Construct an augmented prompt with retrieved document context."""
    context_parts = []
    for i, hit in enumerate(retrieved_docs):
        context_parts.append(
            f"[{i+1}] PMID: {hit.payload['pmid']} | {hit.payload['title']}\n"
            f"{hit.payload['abstract']}\n"
        )
    context = "\n---\n".join(context_parts)

    prompt = f"""You are a clinical assistant grounded in scientific evidence.
Answer the user's query based ONLY on the documents provided below.
Cite sources using [PMID: XXXXX]. If the context is insufficient, say so explicitly.

=== REFERENCE DOCUMENTS ===
{context}
=== END OF DOCUMENTS ===

USER QUERY: {query}

EVIDENCE-BASED RESPONSE:"""

    return prompt


# Build the prompt
augmented_prompt = build_augmented_prompt(query, results.points)

print("Prompt constructed:")
print(f"   Length: {len(augmented_prompt)} characters")
print(f"   Documents in context: {len(results.points)}")
print(f"\n{'=' * 60}")
print(augmented_prompt[:500])
print("...")

In [ ]:
# Option A: Hugging Face Inference API (no GPU required)
# Uses the free HF API. Suitable for this hello world.
# Later phases will use the locally fine-tuned model.

from huggingface_hub import InferenceClient

hf_token = os.getenv("HF_TOKEN")
client = InferenceClient(token=hf_token) if hf_token else InferenceClient()

print("Generating response with LLM...\n")

response = client.text_generation(
    prompt=augmented_prompt,
    model="mistralai/Mistral-7B-Instruct-v0.3",  # Free model for testing
    max_new_tokens=512,
    temperature=0.1,
    repetition_penalty=1.1,
)

print("SYSTEM RESPONSE:\n")
print(response)

In [ ]:
# Option B (alternative): Local model via transformers
# Uncomment if you have a GPU and prefer local inference.
# Uses a small model for this test.

# from transformers import pipeline
#
# generator = pipeline(
#     "text-generation",
#     model="microsoft/Phi-3-mini-4k-instruct",  # Small model for testing
#     device_map="auto",
#     torch_dtype="auto",
# )
#
# output = generator(
#     augmented_prompt,
#     max_new_tokens=512,
#     temperature=0.1,
#     do_sample=True,
# )
#
# print("SYSTEM RESPONSE:\n")
# print(output[0]["generated_text"][len(augmented_prompt):])

## Checkpoint: What have we validated?

If you reached this point without errors, you have confirmed that:

1. **PubMed API** works -- scientific literature can be downloaded
2. **Embedding model** works -- medical text can be converted to vectors
3. **Qdrant** works -- vectors can be stored and searched
4. **Semantic search** works -- queries retrieve relevant documents
5. **Augmented generation** works -- the LLM generates context-grounded responses

### What we have NOT done yet (and that is expected):
- Fine-tuning -- Phase 2
- Intelligent chunking -- Phase 1
- Formal evaluation -- Phase 4
- Optimised prompts -- Phase 3

### Next steps:
- `01_mtsamples_eda.ipynb` -- Explore the training dataset
- `02_pubmed_ingestion.ipynb` -- Full ingestion pipeline
- `03_embedding_benchmark.ipynb` -- Compare embedding models

---

### Decisions made in this notebook

Log in `DECISIONS.md`:
- Initial embedding model: PubMedBERT (benchmark pending)
- PubMed API via Entrez / BioPython
- Qdrant as the vector store
- RAG prompt format (version 0)

## (Bonus) Log to MLflow

Even for this hello world it is worth logging the experiment.
This builds the habit of tracking from the very beginning.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("00-vertical-slice")

with mlflow.start_run(run_name="hello_world_v0"):
    # Parameters
    mlflow.log_param("embedding_model", EMBEDDING_MODEL)
    mlflow.log_param("pubmed_query", "type 2 diabetes treatment guidelines 2024")
    mlflow.log_param("num_papers", len(papers))
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("vector_dim", VECTOR_DIM)

    # Basic metrics (formal evaluation comes in Phase 4)
    avg_score = sum(h.score for h in results.points) / len(results.points)
    mlflow.log_metric("avg_retrieval_score", avg_score)

    # Save the prompt as an artifact
    with open("/tmp/prompt_v0.txt", "w") as f:
        f.write(augmented_prompt)
    mlflow.log_artifact("/tmp/prompt_v0.txt")

    print("Experiment logged in MLflow")
    print(f"   Avg retrieval score: {avg_score:.4f}")
    print(f"   Dashboard: http://localhost:5000")